# Setup
Das Notebook dient dazu die Images zu downloaden.  
Wichtig hierfür sind in der Config unteranderem MAX_WORKER und die Downloadgeschwindigkeit liegt an:
- MAX_WORKERS COUNT
- Rechenleistung
- Internetverbindung
- Cloud Mangagment (emfohlen lokale Speicherung)

In [ ]:
import os
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import pandas as pd
import requests
import yaml
from PIL import Image
from requests.adapters import HTTPAdapter, Retry
from tqdm import tqdm

thread_local = threading.local()

def find_project_root():
    for dir in (Path.cwd(), *Path.cwd().parents):
        if (dir / "config.yaml").exists():
            return dir
    raise FileNotFoundError("Projektroot nicht gefunden")

def find_upwards(name):
    # Erlaubt, das Notebook aus eval/ oder aus dem Projektwurzelverzeichnis
    # zu starten, ohne Pfade anzupassen.
    for d in [Path.cwd(), *Path.cwd().parents]:
        if (d / name).exists():
            return d / name
    return None


cfg_file = find_upwards("config.yaml")
assert cfg_file, "config.yaml nicht gefunden (liegt im Projektwurzelverzeichnis)."
CFG = yaml.safe_load(cfg_file.read_text())


PROJECT_ROOT = find_project_root()
CONFIG_PATH = PROJECT_ROOT / "config.yaml"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
DATA_PATH_META = PROCESSED_DIR / "metadata.parquet"
IMAGE_PATH = (PROJECT_ROOT / CFG["img_download_path"]).expanduser()
IMG_SIZE = CFG["image_size"]
metadata = pd.read_parquet(DATA_PATH_META)
download_metadata = metadata[metadata["split"].isin(["train","database", "query"])].copy()

# Set Max Workers -> Recommendation for new Laptop 96
MAX_WORKERS = CFG["max_workers"]
if MAX_WORKERS == "auto":
    MAX_WORKERS = 96
else:
    MAX_WORKERS = int(MAX_WORKERS)


# Get Token
if env_file := find_upwards(".env"):
    for line in env_file.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            k, v = line.split("=", 1)
            os.environ.setdefault(k.strip(), v.strip().strip("'\""))

TOKEN = os.environ.get("MAPILLARY_TOKEN", "")
assert TOKEN.startswith("MLY|"), ("Kein Mapillary-Token. Datei .env anlegen:\n MAPILLARY_TOKEN=MLY|dein|token")


print("Download Location: ", IMG_PATH)
print(f"Download Threads: {MAX_WORKERS}")
print(f"Zu ladende Bilder: {len(download_metadata):,}")

# API Anfrage
  
Erstelle session, frage API an 

In [ ]:
# Erstelle Session über HTTP Adapter -> https oder http
def make_session():
    session = requests.Session()

    retry = Retry(
        total=5,
        connect=5,
        read=5,
        status=5,
        backoff_factor=1,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=frozenset(["GET"]),
        raise_on_status=True,
        respect_retry_after_header=True,
    )

    adapter = HTTPAdapter(
        max_retries=retry, pool_connections=MAX_WORKERS, pool_maxsize=MAX_WORKERS
    )

    session.mount("https://", adapter)
    session.mount("http://", adapter)

    return session


def get_session():
    if not hasattr(thread_local, "session"):
        thread_local.session = make_session()

    return thread_local.session


def download_image(image_id):

    image_id = str(image_id)
    image_file = IMG_PATH / f"{image_id}.jpg"

    # Bereits vorhandenes, nicht-leeres Bild überspringen
    if image_file.exists() and image_file.stat().st_size > 0:
        return "exists"

    session = get_session()

    try:
        api_url = f"https://graph.mapillary.com/{image_id}"

        params = {
            "fields": f"id,thumb_{IMG_SIZE}_url",
            "access_token": TOKEN,
        }

        api_response = session.get(
            api_url,
            params=params,
            timeout=(10, 30),
        )

        api_response.raise_for_status()

        data = api_response.json()

        image_url = data.get(f"thumb_{IMG_SIZE}_url")

        if not image_url:
            print(f"Keine Bild-URL für {image_id}")
            return "failed"

        image_response = session.get(image_url, timeout=(10, 60))
        image_response.raise_for_status()

        if not image_response.content:
            print(f"Leere Bildantwort für {image_id}")
            return "failed"

        image_file.write_bytes(image_response.content)

        return "downloaded"

    except requests.RequestException as e:
        status = e.response.status_code if e.response is not None else "keine Antwort"

        print(f"Fehler bei {image_id}: {type(e).__name__}: {status}")

        return "failed"

    except (ValueError, KeyError) as e:
        print(f"Ungültige Mapillary-Antwort für {image_id}: {type(e).__name__}")

        return "failed"

    except Exception as e:
        # Fängt unerwartete Fehler ab, damit ein einzelnes Bild
        # nicht den gesamten 98k-Download stoppt.
        print(f"Unerwarteter Fehler bei {image_id}: {type(e).__name__}: {e}")

        return "failed"


# Download

In [ ]:

# Download
results = {}

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {
        executor.submit(download_image, image_id): image_id
        for image_id in download_metadata["image_id"]
    }

    for future in tqdm(as_completed(futures), total=len(futures), desc="Bilder herunterladen"):
        image_id = futures[future]
        try:
            results[image_id] = future.result()
        except Exception as e:
            print(f"Unerwarteter Fehler bei {image_id}: {type(e).__name__}: {e}")
            results[image_id] = "failed"



failed_ids = [i for i, r in results.items() if r == "failed"]
image_files = list(IMG_PATH.glob("*.jpg"))
FAILED_IMG_PATH = PROCESSED_DIR / "failed_image_download.txt"
FAILED_IMG_PATH.write_text("\n".join(map(str, failed_ids)))


# Auswertung erster Download
print("=" * 50)
print("DOWNLOAD ERGEBNISSE")
print("=" * 50)
print(f"Erfolgreiche:               {list(results.values()).count('downloaded'):,}")
print(f"Bereits vorhanden:          {list(results.values()).count('exists'):,}")
print(f"Fehlgeschlagene:            {list(results.values()).count('failed'):,}")
print("=" * 50)
print(f"Fehlgeschalgende Bilder:    {len(failed_ids):,}")
print(f"Gepeicherte failed unter    {FAILED_IMG_PATH}")
print(f"Lokale JPGs:                {len(image_files):,}")
print("=" * 50)


# Kaputte Bilder

In [ ]:
bad = []

for img in IMG_PATH.glob("*.jpg"):
    try:
        with Image.open(img) as im:
            im.verify()
    except Exception:
        bad.append(img)

print("Kaputte Bilder:", len(bad))
print(bad)